# Дообучение LaBraM-Base на NMT → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-26 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Дообучить LaBraM-Base (ICLR 2024, код github.com/935963004/LaBraM) по той же схеме, что REVE, EEG-DINO, CBraMod, BIOT, LUNA и EEGPT; сравнить и собрать в ансамбли с REVE и EEG-DINO. Исправить ошибки блокнота Дарьи `labram-base1.ipynb` (нестрогая загрузка весов без проверки, порог 0,5) |
| **Веса** | `checkpoints/labram-base.pth` из репозитория авторов (коммит `c431221`, 96,6 МБ), модель `labram_base_patch200_200` (5,8 млн параметров). Загрузка как в `run_class_finetuning.py` авторов (`student.*` → модель), но со списками пропущенных и лишних ключей и проверкой: без обучения остаётся только голова (раздел 2) |
| **Предобучение** | ~2500 ч ЭЭГ из ~20 наборов, **в том числе весь TUEG (TUH EEG Corpus)** — а значит, и пациенты TUEP. **Утечка при будущей проверке на TUEP** (как у REVE, CBraMod, EEG-DINO, LUNA) |
| **Вход** | Как у авторов для TUAB: 200 Гц, патчи по 1 с (200 отсчётов), окна 10 с → 19 каналов × 10 патчей, вход в мкВ/100 (`engine_for_finetuning.py`). Каналы — по именам в `standard_1020` из `utils.py` (вложение позиции выбирается по имени, порядок не важен). Для TUAB авторы подают названия TUH **T3/T4/T5/T6** (`run_class_finetuning.py`), в `standard_1020` у них отдельные строки от T7/T8/P7/P8 — берём T3–T6, как у авторов (раздел 2 сравнивает эти строки). Фильтр 0,3–75 Гц + режекция 50 Гц и нормировка окна к общему СКО 15 мкВ — как у EEG-DINO (масштаб NMT в файлах неверный) |
| **Аугментации** | как у EEG-DINO: шум 0,1·СКО, зеркальная замена полушарий (p = 0,5), зануление 1–2 каналов (p = 0,5), сдвиг до 2 с |
| **Обучение** | полное дообучение (вся модель + голова 200 → 1), BCE, AdamW (wd 0,05; без wd — смещения, нормы, `pos_embed`, `cls_token`, `time_embed`, как `no_weight_decay` авторов), **послойное снижение скорости 0,65** (как у авторов для TUAB; номера слоёв — по `get_num_layer_for_vit` авторов), drop path 0,1, разогрев 10 % + косинус, пакет 64, fp16. Базовая скорость — по отложенным 15 % NMT train из 5·10⁻⁵ / 2·10⁻⁴ / 5·10⁻⁴ (5·10⁻⁴ — у авторов для TUAB). До 10 эпох, остановка, если отложенный ROC AUC не растёт 2 эпохи подряд; берётся лучшая эпоха. 3 запуска с выбранной скоростью |
| **Среда** | Kaggle: короткая проверка — процессор Kaggle (`SMOKE = True`, `zakharsann/h1-labram-smoke`), обучение — GPU, две T4 (`zakharsann/h1-labram-finetune`, версия 1). Сеанс чтения без GPU (`zakharsann/h1-labram-prep`) не уложился в срок (>1,5 ч вместо ~20 мин), поэтому данные читались в самом GPU-сеансе (как у EEG-DINO, 18 мин) |

In [ ]:
import os, sys, ast, glob, time, subprocess, threading, traceback, hashlib, warnings
ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = False       # True — короткая проверка на процессоре Kaggle (часть записей, 1 эпоха), тестовые цифры не смотреть
PREP_SAVE = False   # True — сеанс без GPU: сверка весов, чтение данных и сохранение окон, без обучения
LABRAM_COMMIT = "c431221e6cfd23dbfa9950e0180682fb322b0548"   # 935963004/LaBraM, по нему сверялись модель и подготовка
if ON_KAGGLE:
    LABRAM_DIR, OUT = "/kaggle/temp/LaBraM", "/kaggle/working"   # код и веса не в /kaggle/working — не попадут в вывод
    if not os.path.exists(LABRAM_DIR):
        os.makedirs("/kaggle/temp", exist_ok=True)
        subprocess.run(["git", "clone", "-q", "https://github.com/935963004/LaBraM", LABRAM_DIR], check=True)
        subprocess.run(["git", "-C", LABRAM_DIR, "checkout", "-q", LABRAM_COMMIT], check=True)
else:
    LABRAM_DIR, OUT = os.environ["LABRAM_DIR"], "."
sys.path.insert(0, LABRAM_DIR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn.functional as F, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
from modeling_finetune import labram_base_patch200_200
DEV = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
T_NB = time.time()

SFREQ, SEG, PATCH = 200, 10, 200
L = SFREQ * SEG
CAP_TRAIN, CAP_EVAL = 8, 60   # окон по 10 с на запись, как у EEG-DINO, REVE и BIOT

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]

# список каналов авторов без импорта utils.py (там нужен pyhealth)
_src = open(f"{LABRAM_DIR}/utils.py", encoding="utf-8").read()
STANDARD_1020 = next(ast.literal_eval(n.value) for n in ast.walk(ast.parse(_src))
                     if isinstance(n, ast.Assign) and any(getattr(t, "id", "") == "standard_1020" for t in n.targets))
CH_NAMES = [e.upper() for e in EL]                                   # T3/T4/T5/T6 — как ch_names авторов для TUAB
INPUT_CHANS = [0] + [STANDARD_1020.index(c) + 1 for c in CH_NAMES]   # 0 — cls, как utils.get_input_chans
print("каналы:", " ".join(CH_NAMES)); print("input_chans:", INPUT_CHANS)

def labram_raw():
    """Как create_model в run_class_finetuning.py авторов для TUAB (--disable_rel_pos_bias --abs_pos_emb --disable_qkv_bias, drop_path 0,1)."""
    return labram_base_patch200_200(num_classes=1, drop_rate=0.0, drop_path_rate=0.1, attn_drop_rate=0.0, use_mean_pooling=True,
                                    init_scale=0.001, use_rel_pos_bias=False, use_abs_pos_emb=True, init_values=0.1, qkv_bias=False)

def fwd(model, x):
    return model(x, input_chans=INPUT_CHANS).squeeze(-1)

STD_UV, NOISE = 15.0, 0.1
def batch_input(W, g=None):
    """(B, 19, 2000) -> окно к общему СКО 15 мкВ -> мкВ/100 как у авторов -> (аугментации, как у EEG-DINO) -> (B, 19, 10, 200)."""
    x = torch.from_numpy(np.ascontiguousarray(W)).to(DEV).float()
    x = x - x.mean(-1, keepdim=True)
    x = (x / (x.std(dim=(1, 2), keepdim=True) + 1e-6) * (STD_UV / 100.0)).clamp(-5, 5)   # обрезка ±33 СКО — только выбросы
    if g is not None:
        B, C, T = x.shape
        x = x + NOISE * (STD_UV / 100.0) * torch.randn_like(x)
        m = torch.from_numpy(g.random(B) < 0.5).to(x.device)
        x[m] = x[m][:, MIRROR]
        k = np.where(g.random(B) < 0.5, g.integers(1, 3, B), 0)                                  # 0, 1 или 2 зануляемых канала
        x[torch.from_numpy(np.argsort(g.random((B, C)), 1) < k[:, None]).to(x.device)] = 0
        sh = torch.from_numpy(g.integers(-2 * SFREQ, 2 * SFREQ + 1, B)).to(x.device)             # циклический сдвиг до 2 с
        x = x.gather(2, ((torch.arange(T, device=x.device)[None] - sh[:, None]) % T)[:, None, :].expand(B, C, T))
    return x.reshape(len(x), 19, SEG, PATCH)

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")
def predict(model, X, off, bs=256):
    """Медиана вероятности по окнам каждой записи; off — границы записей."""
    model.eval(); p = []
    with torch.no_grad(), torch.autocast(**AMP):
        for b in range(0, len(X), bs):
            p.append(torch.sigmoid(fwd(model, batch_input(X[b:b + bs])).float()).cpu().numpy())
    p = np.concatenate(p)
    return np.array([np.median(p[off[i]:off[i + 1]]) for i in range(len(off) - 1)])

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cb_filter(X):
    """0,3-75 Гц + режекция 50 Гц, как в блокнотах EEG-DINO, CBraMod, REVE и BIOT."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)

def take(X, m):
    """Окна записей из таблицы m (столбцы start, n) подряд + границы записей."""
    idx = np.concatenate([np.arange(s, s + n) for s, n in zip(m["start"], m["n"])])
    return X[idx], np.concatenate([[0], np.cumsum(m["n"].to_numpy())])
print("устройство:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] if DEV == "cuda" else "процессор")

## 2. Проверка весов: ключи чекпоинта ↔ модель авторов

У Дарьи: `student.*` → модель, `load_state_dict(strict=False)` без вывода — расхождения ключей не видны. Здесь печатаются полные списки пропущенных (missing) и лишних (unexpected) ключей при её способе загрузки, затем сопоставление исправляется так, чтобы без обучения осталась **только голова** `head.*`. Ключи чекпоинта, которых нет в модели дообучения (части предобучения), перечисляются явно.

In [ ]:
CKPT = f"{LABRAM_DIR}/checkpoints/labram-base.pth"
sha = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
print(f"файл: {os.path.getsize(CKPT):,} байт, sha256 {sha[:16]}… | верхний уровень: {list(ck.keys()) if isinstance(ck, dict) else type(ck)}")
RAW = ck["model"] if "model" in ck else ck
print("тензоров:", len(RAW), "| по префиксам:", pd.Series([k.split(".")[0] for k in RAW]).value_counts().to_dict())

m0 = labram_raw(); OWN = m0.state_dict()
# 1) способ Дарьи (= авторов): student.* -> без префикса, head.* убрать, strict=False
SD = {k[len("student."):]: v for k, v in RAW.items() if k.startswith("student.")} or dict(RAW)
r0 = labram_raw().load_state_dict({k: v for k, v in SD.items() if not k.startswith("head.")}, strict=False)
print(f"\nспособ Дарьи: пропущено {len(r0.missing_keys)}: {r0.missing_keys}")
print(f"              лишних {len(r0.unexpected_keys)}: {r0.unexpected_keys}")

# 2) исправленное сопоставление: итоговая норма предобучения norm.* -> fc_norm.* (при среднем по токенам norm = Identity,
#    а fc_norm у авторов остаётся новой); ключи, которых нет в модели дообучения, — только части предобучения, отбрасываются явно
MAP = {}
if "fc_norm.weight" in OWN and "norm.weight" not in OWN and "norm.weight" in SD:
    MAP = {"norm.weight": "fc_norm.weight", "norm.bias": "fc_norm.bias"}
SD2 = {MAP.get(k, k): v for k, v in SD.items() if MAP.get(k, k) in OWN and not k.startswith("head.")}
DROPPED = sorted(k for k in SD if MAP.get(k, k) not in OWN)
bad_shape = [k for k in SD2 if SD2[k].shape != OWN[k].shape]
r = m0.load_state_dict(SD2, strict=False)
same = all(torch.equal(m0.state_dict()[k], SD2[k]) for k in SD2)
print(f"\nисправлено: сопоставление {MAP}")
print(f"  пропущено {len(r.missing_keys)}: {r.missing_keys}")
print(f"  лишних {len(r.unexpected_keys)}: {r.unexpected_keys} | расхождений формы: {bad_shape} | значения загружены точно: {same}")
print(f"  не нужны модели дообучения ({len(DROPPED)}): {DROPPED}")
print(f"  загружено {len(SD2)} из {len(OWN)} тензоров модели; параметров {sum(p.numel() for p in m0.parameters()):,}, "
      f"из них без обучения (голова) {sum(OWN[k].numel() for k in r.missing_keys):,}")
HEAD_ONLY = set(r.missing_keys) <= {"head.weight", "head.bias"} and not r.unexpected_keys and not bad_shape and same
assert HEAD_ONLY, "без обучения остаётся не только голова"

# строки pos_embed для старых и новых названий височных каналов (оба варианта есть в standard_1020 и учились при предобучении)
pe = SD["pos_embed"][0]
cos = lambda a, b: float(F.cosine_similarity(pe[STANDARD_1020.index(a) + 1], pe[STANDARD_1020.index(b) + 1], dim=0))
pairs = [("T3", "T7"), ("T4", "T8"), ("T5", "P7"), ("T6", "P8"), ("T3", "T4"), ("T3", "C3"), ("T3", "O2"), ("FP1", "O2")]
COS = {f"{a}~{b}": round(cos(a, b), 3) for a, b in pairs}
print("\nкосинус строк pos_embed:", COS)
print("места в standard_1020: T3/T4/T5/T6 =", [STANDARD_1020.index(c) for c in ["T3", "T4", "T5", "T6"]],
      "| T7/T8/P7/P8 =", [STANDARD_1020.index(c) for c in ["T7", "T8", "P7", "P8"]])

with torch.no_grad():
    z = fwd(m0.eval(), torch.randn(2, 19, SEG, PATCH) * 0.15)
print("выход на случайном входе:", tuple(z.shape), "конечный:", bool(torch.isfinite(z).all()))
rows = [dict(check="коммит репозитория", value=LABRAM_COMMIT), dict(check="sha256 labram-base.pth", value=sha),
        dict(check="тензоров в чекпоинте по префиксам", value=pd.Series([k.split(".")[0] for k in RAW]).value_counts().to_dict()),
        dict(check="способ Дарьи: missing", value=r0.missing_keys), dict(check="способ Дарьи: unexpected", value=r0.unexpected_keys),
        dict(check="исправлено: сопоставление", value=MAP), dict(check="исправлено: missing", value=r.missing_keys),
        dict(check="исправлено: unexpected", value=r.unexpected_keys), dict(check="отброшены (только предобучение)", value=DROPPED),
        dict(check="без обучения только голова", value=HEAD_ONLY), dict(check="косинус строк pos_embed", value=COS),
        dict(check="input_chans (T3–T6 как у авторов для TUAB)", value=INPUT_CHANS)]
pd.DataFrame(rows).to_csv(f"{OUT}/labram_weights_check.csv", index=False)
del m0, ck, RAW

LOCK = threading.Lock()
def new_labram(seed):
    with LOCK:   # torch.manual_seed общий для потоков
        torch.manual_seed(seed)
        m = labram_raw()
        r = m.load_state_dict(SD2, strict=False)
    assert set(r.missing_keys) <= {"head.weight", "head.bias"} and not r.unexpected_keys
    return m.to(DEV)

## 3. NMT и турецкий набор

Код чтения — как у EEG-DINO (те же окна: NMT train до 8 окон по 10 с на запись, NMT eval и Турция — до 60 окон равномерно; Турция: первые 19 каналов (относительно уха) → средний референт, 500 → 200 Гц, куски по 15 с подряд). Код рассчитан на готовые окна из сеанса без GPU (`PREP_SAVE = True`), но в этот раз тот сеанс не закончился вовремя, и чтение шло в GPU-сеансе (ветка `else`, 18 мин).

In [ ]:
import mne
mne.set_log_level("ERROR")
t0 = time.time()
PREP_IN = sorted(glob.glob("/kaggle/input/**/labram_prep_nmt_meta.csv", recursive=True))
if PREP_IN and not PREP_SAVE:
    P = os.path.dirname(PREP_IN[0])
    lab, tmeta = pd.read_csv(f"{P}/labram_prep_nmt_meta.csv"), pd.read_csv(f"{P}/labram_prep_tr_meta.csv")
    NMT_X, TR_X = np.load(f"{P}/labram_prep_nmt.npy"), np.load(f"{P}/labram_prep_tr.npy")
    print(f"готовые окна: NMT {NMT_X.shape}, Турция {TR_X.shape} | {time.time()-t0:.0f} с")
else:
    ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
    ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
    lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
    paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
    lab["path"] = lab["recordname"].map(paths)
    lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
    lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
    tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
             [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
    if SMOKE:
        lab = pd.concat([lab[(lab["loc"] == "train") & (lab.y == 0)].head(40), lab[(lab["loc"] == "train") & (lab.y == 1)].head(20),
                         lab[(lab["loc"] == "eval")].head(20)]).sort_values("recordname").reset_index(drop=True)
        tfiles = tfiles[:10] + tfiles[-10:]
    print(pd.crosstab(lab["loc"], lab["label"]))

    def warm(p):
        with open(p, "rb") as f:
            while f.read(1 << 24):
                pass
    allf = list(lab.path) + [f for f, _ in tfiles]
    with ThreadPoolExecutor(32) as ex:
        for i, _ in enumerate(ex.map(warm, allf)):
            if (i + 1) % 500 == 0 or i + 1 == len(allf):
                print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)

    ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
    TARGET = [e.upper() for e in EL]
    def norm(ch):
        c = ch.upper().replace("EEG", "").strip()
        for s in ["-REF", "-AVG", "-LE", "-AR"]:
            c = c.replace(s, "")
        return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

    NMT_STORE_K = 100.0   # масштаб в файлах неверный; окно всё равно нормируется, множитель только для float16
    def nmt_job(i):
        r = lab.iloc[i]
        try:
            raw = mne.io.read_raw_edf(r.path, preload=True)
            if raw.info["sfreq"] != SFREQ:
                raw.resample(SFREQ)
            names = {norm(c): c for c in raw.ch_names}
            X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
        except Exception:
            return r.recordname, None
        W = windows(cb_filter(X))
        if len(W) == 0:
            return r.recordname, None
        if r["loc"] == "train" and len(W) > CAP_TRAIN:
            W = W[np.sort(np.random.default_rng(i).choice(len(W), CAP_TRAIN, replace=False))]
        elif r["loc"] == "eval":
            W = W[spread(len(W), CAP_EVAL)]
        return r.recordname, W.astype(np.float16)

    NMT = {}
    with ThreadPoolExecutor(4) as ex:
        for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
            if res is not None:
                NMT[name] = res
            if (i + 1) % 400 == 0 or i + 1 == len(lab):
                print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
    lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)[["recordname", "loc", "label", "y"]]
    lab["n"] = [len(NMT[n]) for n in lab.recordname]; lab["start"] = np.concatenate([[0], np.cumsum(lab.n.to_numpy())[:-1]])
    NMT_X = np.concatenate([NMT[n] for n in lab.recordname]); del NMT

    # индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
    IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]
    def tr_job(f):
        d = scipy.io.loadmat(f)["data"]
        X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
        X = X - X.mean(1, keepdims=True)          # средний референт, как у EEG-DINO и CBraMod
        X = ss.resample_poly(X, 2, 5, axis=-1)    # 200 Гц
        W = windows(cb_filter(np.concatenate(list(X), axis=-1)))
        return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))
    TRW, amp = [], []
    with ThreadPoolExecutor(4) as ex:
        for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
            TRW.append(a); amp.append(b)
    tmeta = pd.DataFrame({"id": [os.path.basename(f) for f, _ in tfiles], "y": [yy for _, yy in tfiles], "amp": amp, "n": [len(w) for w in TRW]})
    tmeta["start"] = np.concatenate([[0], np.cumsum(tmeta.n.to_numpy())[:-1]])
    TR_X = np.concatenate(TRW); del TRW
    print(f"прочитано за {time.time()-t0:.0f} с")
    if PREP_SAVE:
        np.save(f"{OUT}/labram_prep_nmt.npy", NMT_X); np.save(f"{OUT}/labram_prep_tr.npy", TR_X)
        lab.to_csv(f"{OUT}/labram_prep_nmt_meta.csv", index=False); tmeta.to_csv(f"{OUT}/labram_prep_tr_meta.csv", index=False)
        print(f"окна сохранены: NMT {NMT_X.shape}, Турция {TR_X.shape}, {(NMT_X.nbytes + TR_X.nbytes)/2**30:.2f} ГБ")

print(pd.crosstab(lab["loc"], lab["label"]))
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
EV_X, EV_OFF = take(NMT_X, ev); TRK_X, TRK_OFF = take(TR_X, tmeta)
y_tr, tr_ids, TR_AMP = tmeta.y.to_numpy(), tmeta.id.tolist(), tmeta.amp.to_numpy()
print("окон по 10 с: NMT train", int(tr.n.sum()), "| NMT eval", len(EV_X), "| Турция", len(TRK_X), f"(на запись: медиана {int(tmeta.n.median())})")
print(f"Турция: эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение LaBraM

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)   # то же деление, что у EEG-DINO
trx = tr.set_index("recordname")
fit_m, val_m = trx.loc[fit_names].reset_index(), trx.loc[val_names].reset_index()
Xw, _ = take(NMT_X, fit_m); yw = np.repeat(fit_m.y.to_numpy(), fit_m.n.to_numpy())
VAL_X, VAL_OFF = take(NMT_X, val_m); y_val = val_m.y.to_numpy()
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор: {len(val_names)} ({len(VAL_X)} окон)")

BS, MAX_EP, PATIENCE, LLRD, WD, NL = 64, (1 if SMOKE else 10), 2, 0.65, 0.05, 12
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()
NO_WD = {"pos_embed", "cls_token", "time_embed"}   # no_weight_decay() модели авторов

def lr_scale(n):
    """Послойное снижение 0,65 по get_num_layer_for_vit авторов: blocks.i -> 0,65^(12-i); cls_token, pos_embed, patch_embed -> 0,65^13;
    остальное (time_embed, fc_norm, голова) -> 1 (у авторов time_embed тоже попадает в последний «слой»)."""
    if n.startswith("blocks."):
        return LLRD ** (NL - int(n.split(".")[1]))
    if n in ("cls_token", "mask_token", "pos_embed") or n.startswith("patch_embed"):
        return LLRD ** (NL + 1)
    return 1.0

def make_opt(model, total_steps, lr):
    groups = {}
    for n, p in model.named_parameters():
        no_wd = p.ndim == 1 or n.endswith(".bias") or n in NO_WD
        groups.setdefault((lr_scale(n), 0.0 if no_wd else WD), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": lr * k, "weight_decay": wd} for (k, wd), ps in groups.items()], betas=(0.9, 0.999))
    warm_n = max(1, int(0.1 * total_steps))   # у авторов 5 из 50 эпох
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t):
    with torch.autocast(**AMP):
        out = fwd(model, x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

def train_run(seed, lr, deadline, tag="labram"):
    """До MAX_EP эпох; остановка, если отложенный ROC AUC не растёт PATIENCE эпох подряд (плато) или кончается время. Возвращается лучшая эпоха."""
    g = np.random.default_rng(seed)
    model = new_labram(seed)
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, MAX_EP * steps_ep, lr)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad, t_ep, stop = -1, None, 0, [], 0, [], "максимум эпох"
    t0 = time.time()
    for ep in range(MAX_EP):
        if t_ep and time.time() + np.mean(t_ep) > deadline:
            stop = "время"; break
        te = time.time(); model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = np.sort(idx[b:b + BS])
            l, bad = step(model, opt, sched, scaler, batch_input(Xw[j], g), torch.tensor(yw[j], device=DEV))
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, VAL_X, VAL_OFF))
        hist.append((round(float(np.nanmean(losses)), 3), round(float(auc), 3))); t_ep.append(time.time() - te)
        print(f"  {tag} lr {lr:g} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | "
              f"шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif ep + 1 - best_ep >= PATIENCE:
            stop = "плато"; break
    print(f"  {tag} lr {lr:g} запуск {seed}: остановка — {stop}, лучшая эпоха {best_ep}", flush=True)
    return dict(state=best_state, best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad, stop=stop, epochs=len(hist),
                min_per_ep=float(np.mean(t_ep)) / 60, minutes=(time.time() - t0) / 60)

In [ ]:
# --- проверка сходимости на малой подвыборке (без аугментаций): потери должны убывать ---
g0 = np.random.default_rng(123)
n_sm, ep_sm = (32, 3) if SMOKE else (128, 8)
sm = np.concatenate([g0.choice(np.where(yw == c)[0], n_sm, replace=False) for c in (0, 1)])
model = new_labram(0); opt, sched = make_opt(model, ep_sm * (len(sm) // BS), 2e-4)
scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
model.train(); sm_losses, tt = [], []
for ep in range(ep_sm):
    g0.shuffle(sm); ls = []
    for b in range(0, len(sm), BS):
        j = np.sort(sm[b:b + BS]); t1 = time.time()
        l, bad = step(model, opt, sched, scaler, batch_input(Xw[j]), torch.tensor(yw[j], device=DEV))
        if DEV == "cuda": torch.cuda.synchronize()
        tt.append(time.time() - t1); ls.append(l)
    sm_losses.append(np.mean(ls)); print(f"  проверка: эпоха {ep+1} потери {sm_losses[-1]:.3f}", flush=True)
t_step = float(np.median(tt[2:])) if len(tt) > 3 else float(np.median(tt))
del model, opt
if DEV == "cuda": torch.cuda.empty_cache()
CONVERGED = bool(np.isfinite(sm_losses).all() and np.mean(sm_losses[-2:]) < np.mean(sm_losses[:2]) - 0.05)
print("сходимость на подвыборке:", "есть" if CONVERGED else f"НЕТ ({np.round(sm_losses, 3)}) — продолжаю, скорость выбирается по отложенным 15 %")
est_ep = (len(yw) / BS * t_step * 1.2 + len(VAL_X) / BS * t_step / 3) / 60
print(f"шаг {t_step:.3f} с | оценка эпохи {est_ep:.1f} мин | GPU: {N_GPU}")

In [ ]:
# --- выбор базовой скорости по отложенным 15 % NMT train (запуск 0), затем запуски 1–2; при двух GPU — по два запуска параллельно ---
TRAIN_MIN = 2 if SMOKE else 38   # срок к 16:45: чтение данных в этом же сеансе ~17 мин, обучение — не больше 38 мин
T_TRAIN = time.time(); DEADLINE = T_TRAIN + TRAIN_MIN * 60
LRS = [2e-4] if SMOKE else [2e-4, 5e-4, 5e-5]
DONE_MIN = []

def run_jobs(jobs, deadline):
    """jobs: (seed, lr, tag). Очередь на N_GPU потоков; задача пропускается, если по уже сделанным не укладывается в срок."""
    res, q = {}, list(jobs)
    def worker(dev):
        while True:
            with LOCK:
                if not q: return
                seed, lr, tag = q.pop(0)
            est = (np.mean(DONE_MIN) if DONE_MIN else 0) * 0.6
            if time.time() + est * 60 > deadline:
                print(f"ПРОПУСК {tag} lr {lr:g} запуск {seed}: не укладывается (оценка {est:.0f} мин)", flush=True); continue
            try:
                ctx = torch.cuda.device(dev) if DEV == "cuda" else threading.Lock()
                with ctx:
                    info = train_run(seed, lr, deadline, tag=f"{tag} gpu{dev}")
                    if DEV == "cuda": torch.cuda.empty_cache()
                res[(seed, lr)] = info; DONE_MIN.append(info["minutes"])
            except Exception:
                traceback.print_exc()
    th = [threading.Thread(target=worker, args=(d,)) for d in range(max(1, min(N_GPU, len(jobs))))]
    for t in th: t.start()
    for t in th: t.join()
    return res

GRID_RES = run_jobs([(0, lr, "выбор") for lr in LRS], DEADLINE - (0 if SMOKE else 15 * 60))
for (s, lr), v in sorted(GRID_RES.items(), key=lambda kv: kv[0][1]):
    print(f"lr {lr:g}: лучшая эпоха {v['best_ep']} из {v['epochs']} ({v['stop']}), отложенные NMT {v['best_auc']:.3f}, {v['minutes']:.1f} мин")
LR = max(GRID_RES, key=lambda k: GRID_RES[k]["best_auc"])[1]
print("выбрана скорость:", LR)
pd.DataFrame([dict(lr=lr, best_epoch=v["best_ep"], epochs=v["epochs"], stop=v["stop"], val=round(v["best_auc"], 3), hist=v["hist"],
                   nan_steps=v["n_bad"], min_per_epoch=round(v["min_per_ep"], 2)) for (s, lr), v in GRID_RES.items()]).to_csv(f"{OUT}/labram_lr_choice.csv", index=False)
FINAL = {0: GRID_RES[(0, LR)]}
FINAL.update({s: v for (s, _), v in run_jobs([(s, LR, "labram") for s in ([] if SMOKE else [1, 2])], DEADLINE).items()})
del GRID_RES

# --- тестовые наборы — один раз, по лучшей эпохе каждого запуска ---
RUNS = []
for seed, info in sorted(FINAL.items()):
    model = new_labram(seed); model.load_state_dict(info["state"])
    s_ev = predict(model, EV_X, EV_OFF); s_tr = predict(model, TRK_X, TRK_OFF); s_val = predict(model, VAL_X, VAL_OFF)
    RUNS.append(dict(model="labram", lr=LR, seed=seed, best_epoch=info["best_ep"], epochs=info["epochs"], stop=info["stop"],
                     val=round(info["best_auc"], 3), hist=info["hist"], nan_steps=info["n_bad"], minutes=round(info["minutes"], 1),
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr, s_val=s_val,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                     n_unique_tr=len(np.unique(s_tr))))
    r = RUNS[-1]
    print(f"labram запуск {seed}: эпоха {r['best_epoch']} из {r['epochs']} ({r['stop']}) | NMT eval {r['nmt_eval']:.3f} | Турция {r['turkey']:.3f} | "
          f"медиана p здоровые/эпилепсия {r['turkey_med_p0']:.3f}/{r['turkey_med_p1']:.3f} | разных значений {r['n_unique_tr']}/{len(s_tr)}", flush=True)
    del model
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr", "s_val")} for r in RUNS])
runs.to_csv(f"{OUT}/labram_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги: ROC AUC, пороговые метрики (порог — по отложенным 15 % NMT train, на NMT eval и Турцию перенесён без изменений)

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": "LaBraM дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "LaBraM дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv(f"{OUT}/labram_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))

# пороговые метрики: оценка — среднее вероятностей запусков (у рангов нет общего масштаба между наборами)
p_val = np.mean([r["s_val"] for r in RUNS], 0); p_ev = np.mean([r["s_ev"] for r in RUNS], 0); p_tr = np.mean([r["s_tr"] for r in RUNS], 0)
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "LaBraM", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train; перенесён", "n": len(y),
                     "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{OUT}/labram_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))

d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": y_ev}); d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr, "amp": TR_AMP})
d3 = pd.DataFrame({"dataset": "nmt_val", "id": list(val_m.recordname), "y": y_val})
for d, k in [(d1, "s_ev"), (d2, "s_tr"), (d3, "s_val")]:
    for r in RUNS: d[f"labram_seed{r['seed']}"] = r[k]
d1["labram_ens_rank"], d2["labram_ens_rank"] = ens_ev, ens_tr
pd.concat([d1, d2, d3]).to_csv(f"{OUT}/labram_finetune_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 6. Вывод Kaggle (GPU, две T4, `zakharsann/h1-labram-finetune`, версия 1; без строк о подкачке и служебных предупреждений)

Перед ним — короткая проверка на процессоре Kaggle (`h1-labram-smoke`, её тестовые цифры не смотрелись). Тестовые наборы считались один раз.

```
каналы: FP1 FP2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 FZ CZ PZ
input_chans: [0, 1, 3, 18, 22, 40, 44, 62, 66, 81, 83, 16, 24, 89, 91, 90, 92, 20, 42, 64]
устройство: ['Tesla T4', 'Tesla T4']
файл: 96,612,769 байт, sha256 7c50583826afac76… | верхний уровень: ['model', 'optimizer', 'epoch', 'args', 'scaler']
тензоров: 229 | по префиксам: {'student': 224, 'lm_head': 2, 'projection_head': 2, 'logit_scale': 1}

способ Дарьи: пропущено 4: ['fc_norm.weight', 'fc_norm.bias', 'head.weight', 'head.bias']
              лишних 5: ['mask_token', 'lm_head.weight', 'lm_head.bias', 'norm.weight', 'norm.bias']

исправлено: сопоставление {'norm.weight': 'fc_norm.weight', 'norm.bias': 'fc_norm.bias'}
  пропущено 2: ['head.weight', 'head.bias']
  лишних 0: [] | расхождений формы: [] | значения загружены точно: True
  не нужны модели дообучения (3): ['lm_head.bias', 'lm_head.weight', 'mask_token']
  загружено 221 из 223 тензоров модели; параметров 5,820,137, из них без обучения (голова) 201

косинус строк pos_embed: {'T3~T7': 0.743, 'T4~T8': 0.67, 'T5~P7': 0.757, 'T6~P8': 0.677, 'T3~T4': 0.135, 'T3~C3': 0.379, 'T3~O2': -0.241, 'FP1~O2': -0.355}
места в standard_1020: T3/T4/T5/T6 = [88, 90, 89, 91] | T7/T8/P7/P8 = [37, 45, 59, 67]
выход на случайном входе: (2,) конечный: True
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
NMT прочитано 400/2417 186 с
NMT прочитано 800/2417 333 с
NMT прочитано 1200/2417 474 с
NMT прочитано 1600/2417 631 с
NMT прочитано 2000/2417 776 с
NMT прочитано 2400/2417 931 с
NMT прочитано 2417/2417 937 с
прочитано за 1059 с
label  abnormal  normal
loc                    
eval         90      95
train       325    1904
окон по 10 с: NMT train 17775 | NMT eval 10406 | Турция 7240 (на запись: медиана 60)
Турция: эпилепсия 50, здоровые 71
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 15103 (патология 0.14) | записей на выбор: 335 (2672 окон)
  проверка: эпоха 1 потери 0.693
  проверка: эпоха 2 потери 0.686
  проверка: эпоха 3 потери 0.680
  проверка: эпоха 4 потери 0.670
  проверка: эпоха 5 потери 0.652
  проверка: эпоха 6 потери 0.653
  проверка: эпоха 7 потери 0.631
  проверка: эпоха 8 потери 0.643
сходимость на подвыборке: есть
шаг 0.236 с | оценка эпохи 1.2 мин | GPU: 2
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 1: потери 0.653 | отложенные NMT 0.769 | шагов с NaN/inf 0 | 61 с
  выбор gpu1 lr 0.0005 запуск 0 | эпоха 1: потери 0.647 | отложенные NMT 0.795 | шагов с NaN/inf 0 | 61 с
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 2: потери 0.609 | отложенные NMT 0.805 | шагов с NaN/inf 0 | 125 с
  выбор gpu1 lr 0.0005 запуск 0 | эпоха 2: потери 0.609 | отложенные NMT 0.804 | шагов с NaN/inf 0 | 125 с
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 3: потери 0.568 | отложенные NMT 0.833 | шагов с NaN/inf 0 | 190 с
  выбор gpu1 lr 0.0005 запуск 0 | эпоха 3: потери 0.541 | отложенные NMT 0.862 | шагов с NaN/inf 0 | 191 с
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 4: потери 0.537 | отложенные NMT 0.847 | шагов с NaN/inf 0 | 255 с
  выбор gpu1 lr 0.0005 запуск 0 | эпоха 4: потери 0.504 | отложенные NMT 0.865 | шагов с NaN/inf 0 | 256 с
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 5: потери 0.502 | отложенные NMT 0.864 | шагов с NaN/inf 0 | 320 с
  выбор gpu1 lr 0.0005 запуск 0 | эпоха 5: потери 0.459 | отложенные NMT 0.834 | шагов с NaN/inf 0 | 321 с
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 6: потери 0.482 | отложенные NMT 0.859 | шагов с NaN/inf 0 | 385 с
  выбор gpu1 lr 0.0005 запуск 0 | эпоха 6: потери 0.429 | отложенные NMT 0.848 | шагов с NaN/inf 0 | 387 с
  выбор gpu1 lr 0.0005 запуск 0: остановка — плато, лучшая эпоха 4
  выбор gpu0 lr 0.0002 запуск 0 | эпоха 7: потери 0.458 | отложенные NMT 0.857 | шагов с NaN/inf 0 | 450 с
  выбор gpu0 lr 0.0002 запуск 0: остановка — плато, лучшая эпоха 5
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 1: потери 0.664 | отложенные NMT 0.733 | шагов с NaN/inf 0 | 66 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 2: потери 0.623 | отложенные NMT 0.766 | шагов с NaN/inf 0 | 131 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 3: потери 0.601 | отложенные NMT 0.782 | шагов с NaN/inf 0 | 196 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 4: потери 0.584 | отложенные NMT 0.784 | шагов с NaN/inf 0 | 262 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 5: потери 0.569 | отложенные NMT 0.800 | шагов с NaN/inf 0 | 327 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 6: потери 0.561 | отложенные NMT 0.806 | шагов с NaN/inf 0 | 392 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 7: потери 0.551 | отложенные NMT 0.809 | шагов с NaN/inf 0 | 458 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 8: потери 0.546 | отложенные NMT 0.814 | шагов с NaN/inf 0 | 523 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 9: потери 0.542 | отложенные NMT 0.814 | шагов с NaN/inf 0 | 588 с
  выбор gpu1 lr 5e-05 запуск 0 | эпоха 10: потери 0.534 | отложенные NMT 0.814 | шагов с NaN/inf 0 | 654 с
  выбор gpu1 lr 5e-05 запуск 0: остановка — плато, лучшая эпоха 8
lr 5e-05: лучшая эпоха 8 из 10 (плато), отложенные NMT 0.814, 10.9 мин
lr 0.0002: лучшая эпоха 5 из 7 (плато), отложенные NMT 0.864, 7.5 мин
lr 0.0005: лучшая эпоха 4 из 6 (плато), отложенные NMT 0.865, 6.4 мин
выбрана скорость: 0.0005
  labram gpu0 lr 0.0005 запуск 1 | эпоха 1: потери 0.649 | отложенные NMT 0.778 | шагов с NaN/inf 0 | 61 с
  labram gpu1 lr 0.0005 запуск 2 | эпоха 1: потери 0.646 | отложенные NMT 0.764 | шагов с NaN/inf 0 | 66 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 2: потери 0.598 | отложенные NMT 0.801 | шагов с NaN/inf 0 | 124 с
  labram gpu1 lr 0.0005 запуск 2 | эпоха 2: потери 0.593 | отложенные NMT 0.833 | шагов с NaN/inf 0 | 131 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 3: потери 0.553 | отложенные NMT 0.828 | шагов с NaN/inf 0 | 187 с
  labram gpu1 lr 0.0005 запуск 2 | эпоха 3: потери 0.533 | отложенные NMT 0.832 | шагов с NaN/inf 0 | 197 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 4: потери 0.509 | отложенные NMT 0.845 | шагов с NaN/inf 0 | 251 с
  labram gpu1 lr 0.0005 запуск 2 | эпоха 4: потери 0.497 | отложенные NMT 0.848 | шагов с NaN/inf 0 | 263 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 5: потери 0.469 | отложенные NMT 0.865 | шагов с NaN/inf 0 | 315 с
  labram gpu1 lr 0.0005 запуск 2 | эпоха 5: потери 0.451 | отложенные NMT 0.841 | шагов с NaN/inf 0 | 328 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 6: потери 0.436 | отложенные NMT 0.853 | шагов с NaN/inf 0 | 380 с
  labram gpu1 lr 0.0005 запуск 2 | эпоха 6: потери 0.421 | отложенные NMT 0.844 | шагов с NaN/inf 0 | 394 с
  labram gpu1 lr 0.0005 запуск 2: остановка — плато, лучшая эпоха 4
  labram gpu0 lr 0.0005 запуск 1 | эпоха 7: потери 0.406 | отложенные NMT 0.875 | шагов с NaN/inf 0 | 445 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 8: потери 0.388 | отложенные NMT 0.873 | шагов с NaN/inf 0 | 509 с
  labram gpu0 lr 0.0005 запуск 1 | эпоха 9: потери 0.360 | отложенные NMT 0.868 | шагов с NaN/inf 0 | 574 с
  labram gpu0 lr 0.0005 запуск 1: остановка — плато, лучшая эпоха 7
labram запуск 0: эпоха 4 из 6 (плато) | NMT eval 0.832 | Турция 0.750 | медиана p здоровые/эпилепсия 0.058/0.316 | разных значений 119/121
labram запуск 1: эпоха 7 из 9 (плато) | NMT eval 0.840 | Турция 0.863 | медиана p здоровые/эпилепсия 0.011/0.044 | разных значений 113/121
labram запуск 2: эпоха 4 из 6 (плато) | NMT eval 0.862 | Турция 0.911 | медиана p здоровые/эпилепсия 0.020/0.313 | разных значений 114/121
всё обучение: 28.4 мин
 model  lr  seed  best_epoch  epochs  stop   val  nan_steps  minutes  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
labram 0.0     0           4       6 плато 0.865          0      6.4     0.832   0.750          0.058          0.316          119
labram 0.0     1           7       9 плато 0.875          0      9.6     0.840   0.863          0.011          0.044          113
labram 0.0     2           4       6 плато 0.848          0      6.6     0.862   0.911          0.020          0.313          114
                               модель n          NMT eval                             Турция
          LaBraM дообученная, запуски 3     0.845 ± 0.015 0.841 ± 0.083 (0.75 / 0.86 / 0.91)
LaBraM дообученная, ансамбль запусков   0.852 (0.80-0.90)                  0.869 (0.80-0.93)
Связь с амплитудой записи на Турции (Спирмен): 0.29
модель                 порог  значение порога    набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
LaBraM                  Юден            0.457 NMT eval 185 0.77 (0.71-0.83)  0.77 (0.71-0.83) 0.77 (0.68-0.86) 0.76 (0.66-0.84) 0.79 (0.70-0.87)
LaBraM                  Юден            0.457   Турция 121 0.67 (0.59-0.75)  0.60 (0.55-0.66) 1.00 (1.00-1.00) 0.20 (0.09-0.31) 1.00 (1.00-1.00)
LaBraM чувствительность 80 %            0.493 NMT eval 185 0.77 (0.71-0.83)  0.77 (0.70-0.83) 0.78 (0.69-0.87) 0.72 (0.63-0.81) 0.81 (0.73-0.89)
LaBraM чувствительность 80 %            0.493   Турция 121 0.65 (0.57-0.74)  0.58 (0.53-0.63) 1.00 (1.00-1.00) 0.16 (0.06-0.26) 1.00 (1.00-1.00)

всё время блокнота: 46.3 мин
```

## 7. Сравнение с другими моделями, ансамбли, калибровка на месте (локально, без GPU, по сохранённым предсказаниям)

Запускать из папки блокнота: читает `results/labram_finetune_per_person.csv` и `results/eegdino_ensembles_per_person.csv` (общие записи NMT eval 184 + Турция 121 и ранги REVE, EEG-DINO, CBraMod, BIOT, LUNA, DeepEpilepsy; BIOT — также в `biot_ensembles_per_person.csv`). Калибровка — та же функция и те же 500 выборок (seed 0, тот же порядок людей), что в `eegdino_turkey_local_threshold_calibration.csv`; совпадение проверяется по строкам прежних моделей.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr

RES = "../results"
lb = pd.read_csv(f"{RES}/labram_finetune_per_person.csv")
lu = pd.read_csv(f"{RES}/eegdino_ensembles_per_person.csv")   # общие записи и ранги прежних моделей, включая EEG-DINO и REVE
lb_cols = [c for c in lb.columns if c.startswith("labram_seed")]
m = lu.merge(lb[["dataset", "id", "y", "amp"] + lb_cols], on=["dataset", "id"], how="left", suffixes=("", "_l"))
assert m[lb_cols].notna().all().all() and (m.y == m.y_l).all() and len(m) == len(lu)
print({d: int((m.dataset == d).sum()) for d in ["nmt_eval", "turkey"]}, "общих записей")

rk = lambda s: rankdata(s) / len(s)
SC = {}
for ds in ["nmt_eval", "turkey"]:
    d = m[m.dataset == ds]
    s = {"LaBraM": np.mean([rk(d[c]) for c in lb_cols], 0), "REVE": rk(d.reve_rank), "EEG-DINO": rk(d.eegdino_rank), "CBraMod": rk(d.cbramod_rank),
         "BIOT": rk(d.biot_rank), "LUNA": rk(d.luna_rank), "DeepEpilepsy": rk(d.deepepilepsy_rank)}
    s["LaBraM + REVE"] = (s["LaBraM"] + s["REVE"]) / 2
    s["LaBraM + EEG-DINO + REVE"] = (s["LaBraM"] + s["EEG-DINO"] + s["REVE"]) / 3
    s["EEG-DINO + REVE"] = d.eegdino_reve_rank.to_numpy()   # готовый столбец, как в eegdino_turkey_local_threshold_calibration.csv
    SC[ds] = (d.y.to_numpy(), s, d)

def boot(y, f, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(f(i))
    return np.percentile(v, [2.5, 97.5])

rows = []
for name in SC["turkey"][1]:
    row = {"модель / ансамбль (средние ранги)": name}
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]; lo, hi = boot(y, lambda i: roc_auc_score(y[i], s[name][i]))
        row[lab] = f"{roc_auc_score(y, s[name]):.3f} ({lo:.2f}-{hi:.2f})"
    rows.append(row)
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))

print("\nПарный бутстреп разности ROC AUC (95% ДИ):")
diffs = []
for a, b in [("LaBraM", "REVE"), ("LaBraM", "EEG-DINO"), ("LaBraM", "CBraMod"), ("LaBraM", "BIOT"), ("LaBraM", "LUNA"), ("LaBraM", "DeepEpilepsy"),
             ("LaBraM + REVE", "REVE"), ("LaBraM + EEG-DINO + REVE", "REVE"), ("LaBraM + EEG-DINO + REVE", "EEG-DINO + REVE")]:
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]
        f = lambda i: roc_auc_score(y[i], s[a][i]) - roc_auc_score(y[i], s[b][i])
        lo, hi = boot(y, f); dv = f(np.arange(len(y)))
        diffs.append({"разность": f"{a} − {b}", "набор": lab, "ΔROC AUC": round(dv, 3), "ДИ": f"{lo:+.3f}…{hi:+.3f}"})
        print(f"  {a} − {b} | {lab}: {dv:+.3f} ({lo:+.3f}…{hi:+.3f})")
y, s, d = SC["turkey"]
others = ["REVE", "EEG-DINO", "CBraMod", "BIOT", "LUNA", "DeepEpilepsy"]
print("\nСпирмен на Турции: LaBraM–амплитуда", round(spearmanr(s["LaBraM"], d.amp).correlation, 2),
      "| " + " | ".join(f"LaBraM–{n} {spearmanr(s['LaBraM'], s[n]).correlation:.2f}" for n in others))
print("амплитуда у других (Спирмен):", {n: round(float(spearmanr(s[n], d.amp).correlation), 2) for n in others})
print("одна амплитуда как оценка, ROC AUC:", round(roc_auc_score(y, d.amp), 3))

# --- калибровка на месте: порог по k размеченным людям Турции (поровну классов, индекс Юдена), проверка на остальных; 500 повторов, одни и те же выборки для всех моделей ---
def calib(y, sc, k, reps=500, seed=0):
    r = np.random.default_rng(seed); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]; M = {n: [] for n in sc}
    for _ in range(reps):
        c = np.concatenate([r.choice(i1, k // 2, replace=False), r.choice(i0, k // 2, replace=False)])
        rest = np.setdiff1d(np.arange(len(y)), c); yy = y[rest]
        for n, s in sc.items():
            fpr, tpr, th = roc_curve(y[c], s[c]); pr = s[rest] >= th[np.argmax(tpr - fpr)]
            tp = (pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum(); fn = (~pr & (yy == 1)).sum()
            rec, spec = tp / (tp + fn), tn / (tn + fp)
            M[n].append(dict(rec=rec, spec=spec, bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(yy)))
    out = []
    for n in sc:
        D = pd.DataFrame(M[n]); f = lambda c: f"{D[c].mean():.2f} ({np.nanpercentile(D[c], 2.5):.2f}–{np.nanpercentile(D[c], 97.5):.2f})"
        out.append({"калибровка": f"{k} чел.", "модель": n, "recall": f("rec"), "специфичность": f("spec"), "сбаланс. accuracy": f("bacc"),
                    "precision": f("prec"), "accuracy": f("acc")})
    return out
cal_models = ["LaBraM", "REVE", "EEG-DINO", "CBraMod", "BIOT", "LUNA", "DeepEpilepsy", "EEG-DINO + REVE", "LaBraM + REVE", "LaBraM + EEG-DINO + REVE"]
cal = pd.DataFrame([row for k in [10, 20, 30] for row in calib(y, {n: s[n] for n in cal_models}, k)])
old = pd.read_csv(f"{RES}/eegdino_turkey_local_threshold_calibration.csv")
chk = cal.merge(old, on=["калибровка", "модель"], suffixes=("", "_old"))
same = all((chk[c] == chk[c + "_old"]).all() for c in ["recall", "специфичность", "сбаланс. accuracy", "precision", "accuracy"])
print(f"\nвыборки совпадают с eegdino_turkey_local_threshold_calibration.csv: {same} (сверено {len(chk)} строк старых моделей)")
assert same
print("Калибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):")
print(cal.to_string(index=False))

summ.to_csv(f"{RES}/labram_ensembles_summary.csv", index=False)
pd.DataFrame(diffs).to_csv(f"{RES}/labram_paired_differences.csv", index=False)
cal.to_csv(f"{RES}/labram_turkey_local_threshold_calibration.csv", index=False)
pp = []
for ds in ["nmt_eval", "turkey"]:
    yy, s, d = SC[ds]; t = d[["dataset", "id", "y"]].copy()
    for n, v in s.items(): t[n.replace(" + ", "_").replace(" ", "").replace("-", "").lower() + "_rank"] = v
    pp.append(t)
pd.concat(pp).to_csv(f"{RES}/labram_ensembles_per_person.csv", index=False)

Вывод (локально):

```
{'nmt_eval': 184, 'turkey': 121} общих записей
модель / ансамбль (средние ранги)          NMT eval            Турция
                           LaBraM 0.856 (0.80-0.90) 0.869 (0.80-0.93)
                             REVE 0.879 (0.83-0.92) 0.874 (0.80-0.93)
                         EEG-DINO 0.878 (0.83-0.92) 0.856 (0.79-0.91)
                          CBraMod 0.845 (0.79-0.90) 0.820 (0.74-0.88)
                             BIOT 0.798 (0.73-0.86) 0.821 (0.73-0.89)
                             LUNA 0.858 (0.80-0.91) 0.795 (0.71-0.87)
                     DeepEpilepsy 0.900 (0.85-0.94) 0.682 (0.59-0.77)
                    LaBraM + REVE 0.880 (0.83-0.92) 0.895 (0.83-0.95)
         LaBraM + EEG-DINO + REVE 0.885 (0.84-0.93) 0.898 (0.83-0.95)
                  EEG-DINO + REVE 0.893 (0.84-0.93) 0.896 (0.83-0.95)

Парный бутстреп разности ROC AUC (95% ДИ):
  LaBraM − REVE | NMT eval: -0.023 (-0.064…+0.017)
  LaBraM − REVE | Турция: -0.005 (-0.063…+0.052)
  LaBraM − EEG-DINO | NMT eval: -0.022 (-0.050…+0.006)
  LaBraM − EEG-DINO | Турция: +0.013 (-0.031…+0.058)
  LaBraM − CBraMod | NMT eval: +0.011 (-0.022…+0.044)
  LaBraM − CBraMod | Турция: +0.049 (-0.010…+0.113)
  LaBraM − BIOT | NMT eval: +0.058 (+0.015…+0.103)
  LaBraM − BIOT | Турция: +0.048 (-0.026…+0.124)
  LaBraM − LUNA | NMT eval: -0.002 (-0.050…+0.044)
  LaBraM − LUNA | Турция: +0.073 (+0.006…+0.148)
  LaBraM − DeepEpilepsy | NMT eval: -0.044 (-0.089…-0.000)
  LaBraM − DeepEpilepsy | Турция: +0.187 (+0.087…+0.280)
  LaBraM + REVE − REVE | NMT eval: +0.001 (-0.018…+0.021)
  LaBraM + REVE − REVE | Турция: +0.021 (-0.006…+0.051)
  LaBraM + EEG-DINO + REVE − REVE | NMT eval: +0.006 (-0.021…+0.031)
  LaBraM + EEG-DINO + REVE − REVE | Турция: +0.024 (-0.011…+0.062)
  LaBraM + EEG-DINO + REVE − EEG-DINO + REVE | NMT eval: -0.008 (-0.019…+0.002)
  LaBraM + EEG-DINO + REVE − EEG-DINO + REVE | Турция: +0.002 (-0.014…+0.020)

Спирмен на Турции: LaBraM–амплитуда 0.29 | LaBraM–REVE 0.68 | LaBraM–EEG-DINO 0.84 | LaBraM–CBraMod 0.69 | LaBraM–BIOT 0.66 | LaBraM–LUNA 0.68 | LaBraM–DeepEpilepsy 0.38
амплитуда у других (Спирмен): {'REVE': 0.31, 'EEG-DINO': 0.24, 'CBraMod': 0.12, 'BIOT': 0.25, 'LUNA': 0.19, 'DeepEpilepsy': 0.12}
одна амплитуда как оценка, ROC AUC: 0.764

выборки совпадают с eegdino_turkey_local_threshold_calibration.csv: True (сверено 21 строк старых моделей)
Калибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):
калибровка                   модель           recall    специфичность сбаланс. accuracy        precision         accuracy
   10 чел.                   LaBraM 0.63 (0.20–0.96) 0.83 (0.50–1.00)  0.73 (0.60–0.78) 0.78 (0.57–1.00) 0.75 (0.67–0.78)
   10 чел.                     REVE 0.62 (0.24–0.91) 0.88 (0.50–1.00)  0.75 (0.62–0.81) 0.85 (0.55–1.00) 0.78 (0.66–0.85)
   10 чел.                 EEG-DINO 0.64 (0.22–0.96) 0.80 (0.56–1.00)  0.72 (0.61–0.78) 0.73 (0.59–1.00) 0.74 (0.68–0.77)
   10 чел.                  CBraMod 0.61 (0.18–0.96) 0.76 (0.36–1.00)  0.69 (0.59–0.74) 0.70 (0.51–1.00) 0.70 (0.60–0.74)
   10 чел.                     BIOT 0.59 (0.14–0.92) 0.83 (0.36–0.98)  0.71 (0.56–0.77) 0.77 (0.49–0.92) 0.73 (0.59–0.80)
   10 чел.                     LUNA 0.55 (0.13–0.89) 0.82 (0.36–1.00)  0.69 (0.56–0.73) 0.77 (0.49–1.00) 0.71 (0.59–0.77)
   10 чел.             DeepEpilepsy 0.54 (0.02–0.94) 0.65 (0.22–1.00)  0.59 (0.51–0.65) 0.55 (0.45–0.80) 0.60 (0.52–0.64)
   10 чел.          EEG-DINO + REVE 0.65 (0.27–0.93) 0.90 (0.55–1.00)  0.77 (0.63–0.83) 0.86 (0.58–1.00) 0.80 (0.67–0.86)
   10 чел.            LaBraM + REVE 0.64 (0.28–0.93) 0.91 (0.59–1.00)  0.77 (0.63–0.82) 0.88 (0.60–1.00) 0.80 (0.68–0.85)
   10 чел. LaBraM + EEG-DINO + REVE 0.66 (0.24–0.93) 0.89 (0.63–1.00)  0.77 (0.62–0.83) 0.85 (0.63–1.00) 0.79 (0.69–0.86)
   20 чел.                   LaBraM 0.68 (0.30–0.95) 0.80 (0.49–1.00)  0.74 (0.65–0.79) 0.73 (0.55–1.00) 0.75 (0.67–0.79)
   20 чел.                     REVE 0.65 (0.38–0.90) 0.88 (0.49–1.00)  0.77 (0.66–0.82) 0.84 (0.54–1.00) 0.79 (0.65–0.86)
   20 чел.                 EEG-DINO 0.72 (0.30–0.96) 0.75 (0.54–0.98)  0.74 (0.63–0.78) 0.68 (0.57–0.92) 0.74 (0.69–0.78)
   20 чел.                  CBraMod 0.66 (0.25–0.95) 0.73 (0.39–1.00)  0.69 (0.61–0.74) 0.66 (0.51–1.00) 0.70 (0.61–0.74)
   20 чел.                     BIOT 0.64 (0.34–0.88) 0.82 (0.43–0.98)  0.73 (0.63–0.78) 0.75 (0.50–0.93) 0.75 (0.60–0.81)
   20 чел.                     LUNA 0.56 (0.25–0.93) 0.82 (0.34–1.00)  0.69 (0.61–0.74) 0.76 (0.48–1.00) 0.72 (0.57–0.78)
   20 чел.             DeepEpilepsy 0.64 (0.10–0.95) 0.57 (0.25–0.97)  0.60 (0.52–0.66) 0.51 (0.44–0.71) 0.60 (0.52–0.65)
   20 чел.          EEG-DINO + REVE 0.70 (0.42–0.90) 0.89 (0.57–0.98)  0.79 (0.69–0.84) 0.84 (0.59–0.97) 0.81 (0.71–0.87)
   20 чел.            LaBraM + REVE 0.69 (0.42–0.93) 0.88 (0.54–1.00)  0.78 (0.70–0.83) 0.84 (0.56–1.00) 0.80 (0.68–0.85)
   20 чел. LaBraM + EEG-DINO + REVE 0.70 (0.42–0.93) 0.87 (0.62–1.00)  0.79 (0.71–0.84) 0.82 (0.61–1.00) 0.81 (0.73–0.86)
   30 чел.                   LaBraM 0.71 (0.36–0.94) 0.77 (0.50–1.00)  0.74 (0.67–0.80) 0.70 (0.54–1.00) 0.75 (0.66–0.80)
   30 чел.                     REVE 0.67 (0.43–0.91) 0.87 (0.46–1.00)  0.77 (0.68–0.83) 0.82 (0.51–1.00) 0.79 (0.63–0.87)
   30 чел.                 EEG-DINO 0.75 (0.40–0.97) 0.73 (0.54–0.95)  0.74 (0.66–0.79) 0.65 (0.56–0.82) 0.74 (0.68–0.79)
   30 чел.                  CBraMod 0.68 (0.26–0.94) 0.72 (0.51–1.00)  0.70 (0.62–0.76) 0.64 (0.53–1.00) 0.70 (0.65–0.76)
   30 чел.                     BIOT 0.64 (0.40–0.86) 0.82 (0.62–0.98)  0.73 (0.66–0.79) 0.74 (0.57–0.93) 0.75 (0.69–0.81)
   30 чел.                     LUNA 0.56 (0.29–0.91) 0.83 (0.36–1.00)  0.69 (0.63–0.75) 0.76 (0.47–1.00) 0.73 (0.57–0.80)
   30 чел.             DeepEpilepsy 0.66 (0.14–0.94) 0.55 (0.23–0.95)  0.61 (0.52–0.67) 0.49 (0.42–0.65) 0.59 (0.50–0.65)
   30 чел.          EEG-DINO + REVE 0.72 (0.49–0.91) 0.87 (0.65–0.98)  0.80 (0.72–0.84) 0.81 (0.61–0.96) 0.81 (0.73–0.87)
   30 чел.            LaBraM + REVE 0.70 (0.47–0.91) 0.87 (0.55–1.00)  0.79 (0.71–0.83) 0.81 (0.56–1.00) 0.80 (0.68–0.87)
   30 чел. LaBraM + EEG-DINO + REVE 0.74 (0.49–0.94) 0.85 (0.62–0.98)  0.79 (0.73–0.84) 0.79 (0.59–0.96) 0.80 (0.73–0.87)
```

## 8. Итоги

### Проверка весов (`results/labram_weights_check.csv`)
- Чекпоинт `labram-base.pth` (коммит `c431221`, 96,6 МБ, sha256 `7c505838…`): `model` = 224 тензора `student.*` + `lm_head` (2), `projection_head` (2), `logit_scale` — части предобучения.
- **Ошибка в блокноте Дарьи**: при её загрузке (`student.*`, `strict=False`) пропущено 4 ключа — не только голова `head.*`, но и **`fc_norm.weight/bias`** (итоговая норма оставалась новой); лишние — `mask_token`, `lm_head.*` и `norm.*`. Причина: при среднем по токенам в модели дообучения `norm` = Identity, а итоговая норма называется `fc_norm`.
- **Исправлено**: `norm.* → fc_norm.*`. Теперь пропущены только `head.weight`, `head.bias` (201 параметр из 5 820 137), лишних 0, расхождений формы 0, значения загружены точно. Отброшены только `lm_head.*` и `mask_token` (нужны лишь при предобучении).
- Каналы: у авторов для TUAB — названия TUH T3/T4/T5/T6, в `standard_1020` это отдельные строки (88–91), не T7/T8/P7/P8 (37, 45, 59, 67). Косинус строк `pos_embed` T3~T7 0,74, T4~T8 0,67, T5~P7 0,76, T6~P8 0,68 (для сравнения T3~T4 0,14): близкие, но не одинаковые. Взяты T3–T6, как у авторов.

### Выбор скорости и эпохи (`labram_lr_choice.csv`)
| Скорость | Лучшая эпоха | Отложенные 15 % NMT |
|---|---|---|
| 5·10⁻⁵ | 8 из 10 | 0,814 |
| 2·10⁻⁴ | 5 из 7 | 0,864 |
| **5·10⁻⁴** (у авторов) | **4 из 6** | **0,865** |

Все запуски остановились по плато («2 эпохи без роста»). Запуски с 5·10⁻⁴: лучшие эпохи 4 / 7 / 4.

### ROC AUC (95% ДИ, бутстреп по людям; общие записи: NMT eval 184 + Турция 121)

| Модель / ансамбль (средние ранги) | NMT eval | Турция |
|---|---|---|
| **LaBraM (3 запуска)** | 0,856 (0,80–0,90) | **0,869 (0,80–0,93)** |
| REVE | 0,879 (0,83–0,92) | 0,874 (0,80–0,93) |
| EEG-DINO | 0,878 (0,83–0,92) | 0,856 (0,79–0,91) |
| CBraMod | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| LaBraM + REVE | 0,880 (0,83–0,92) | 0,895 (0,83–0,95) |
| **LaBraM + EEG-DINO + REVE** | 0,885 (0,84–0,93) | **0,898 (0,83–0,95)** |
| EEG-DINO + REVE (для сравнения) | 0,893 (0,84–0,93) | 0,896 (0,83–0,95) |

Запуски LaBraM: NMT eval 0,832 / 0,840 / 0,862, Турция **0,750 / 0,863 / 0,911** — большой разброс между запусками. На всех 185 записях NMT eval ансамбль запусков 0,852 (0,80–0,90).

Парные разности ROC AUC (Турция; NMT eval), ни одна с REVE / EEG-DINO / CBraMod не значима:
- LaBraM − REVE: −0,005 (−0,063…+0,052); NMT eval −0,023 (−0,064…+0,017).
- LaBraM − EEG-DINO: +0,013 (−0,031…+0,058); NMT eval −0,022 (−0,050…+0,006).
- LaBraM − CBraMod: +0,049 (−0,010…+0,113); NMT eval +0,011.
- LaBraM + REVE − REVE: +0,021 (−0,006…+0,051); NMT eval +0,001.
- LaBraM + EEG-DINO + REVE − REVE: +0,024 (−0,011…+0,062); − (EEG-DINO + REVE): +0,003 (−0,012…+0,019).
- Значимо: LaBraM лучше LUNA на Турции (+0,073) и BIOT на NMT eval (+0,058); DeepEpilepsy на Турции (+0,187).

### Пороговые метрики (среднее вероятностей 3 запусков; порог по отложенным 15 % NMT train, без изменений перенесён)

| Порог | Набор | Accuracy | Сбаланс. accuracy | Precision | Recall | Специфичность |
|---|---|---|---|---|---|---|
| Юден (0,457) | NMT eval (185) | 0,77 (0,71–0,83) | 0,77 (0,71–0,83) | 0,77 (0,68–0,86) | 0,76 (0,66–0,84) | 0,79 (0,70–0,87) |
| то же | Турция (121) | 0,67 (0,59–0,75) | 0,60 (0,55–0,66) | 1,00 | **0,20 (0,09–0,31)** | 1,00 |
| чувств. 80 % (0,493) | NMT eval | 0,77 (0,71–0,83) | 0,77 (0,70–0,83) | 0,78 (0,69–0,87) | 0,72 (0,63–0,81) | 0,81 (0,73–0,89) |
| то же | Турция | 0,65 (0,57–0,74) | 0,58 (0,53–0,63) | 1,00 | 0,16 (0,06–0,26) | 1,00 |

### Калибровка на месте (Турция: порог по k людям поровну классов, Юден; 500 повторов, те же выборки, что в `eegdino_turkey_local_threshold_calibration.csv` — совпадение строк прежних моделей проверено)

Сбалансированная accuracy:

| k | LaBraM | REVE | EEG-DINO | CBraMod | EEG-DINO + REVE | LaBraM + REVE | LaBraM + EEG-DINO + REVE |
|---|---|---|---|---|---|---|---|
| 10 | 0,73 (0,60–0,78) | 0,75 | 0,72 | 0,69 | 0,77 | 0,77 | 0,77 |
| 20 | **0,74 (0,65–0,79)** | 0,77 | 0,74 | 0,69 | 0,79 | 0,78 | 0,79 |
| 30 | 0,74 (0,67–0,80) | 0,77 | 0,74 | 0,70 | 0,80 | 0,79 | 0,79 |

При 20 людях у LaBraM: recall 0,68, специфичность 0,80, precision 0,73, accuracy 0,75.

## 9. Выводы

1. **LaBraM переносится на Турцию на уровне REVE и EEG-DINO**: 0,869 (0,80–0,93) против 0,874 и 0,856; разности не значимы. Внутри NMT немного слабее (0,856 против 0,88; не значимо).
2. **Запуски сильно различаются на Турции** (0,75 / 0,86 / 0,91) при близком отложенном AUC NMT (0,865 / 0,875 / 0,848). Значит, по NMT нельзя заранее выбрать «удачный» запуск — нужен ансамбль запусков.
3. **Ансамбли**: LaBraM + EEG-DINO + REVE даёт 0,898 на Турции (лучшее значение среди наших), но это всего +0,003 к EEG-DINO + REVE и +0,024 к одной REVE — не значимо. Ранги LaBraM сильно связаны с EEG-DINO (Спирмен 0,84), поэтому нового она добавляет мало.
4. **Перенесённый порог не работает**: на Турции вероятности сдвинуты вниз (медиана p у здоровых 0,01–0,06), и при пороге с NMT чувствительность 0,16–0,20 при специфичности 1,0. Как и у REVE, CBraMod, BIOT, LUNA. При калибровке на месте по 20 людям сбалансированная accuracy 0,74 — как у EEG-DINO, ниже REVE (0,77).
5. **Амплитуда**: Спирмен с СКО записи +0,29 — на уровне REVE (+0,31), выше EEG-DINO (+0,24). Одна амплитуда даёт AUC 0,76, модель — 0,87.
6. **Утечка для TUEP**: LaBraM предобучена на TUEG (весь TUH EEG Corpus), а TUEP — его часть. При будущей проверке на TUEP пациенты почти наверняка были в предобучении; результат на TUEP без исключения этих пациентов будет завышен (то же у REVE, CBraMod, EEG-DINO, LUNA; без TUH — только EEGPT).
7. **Ограничения**: 3 запуска при большом разбросе; в Турции 121 человек, ДИ широкие; NMT («патология») и Турция («эпилепсия против здоровых») — разные задачи. В обучающей части NMT прочиталась 2229 записей (у EEG-DINO та же подготовка), отложенные 15 % делились тем же правилом. Дообученные веса не сохранялись.

**Затраты GPU:** один сеанс, 46,3 мин работы блокнота (чтение данных 18 мин + обучение 28 мин, две T4: скорости и запуски шли по два параллельно) — **≈0,8 ч** из бюджета 1,5 ч. Короткая проверка и неудавшийся сеанс подготовки шли на процессоре Kaggle и GPU не тратили.

Файлы: `results/labram_weights_check.csv`, `labram_lr_choice.csv`, `labram_finetune_{runs,summary,per_person}.csv`, `labram_threshold_metrics.csv`, `labram_ensembles_{summary,per_person}.csv`, `labram_paired_differences.csv`, `labram_turkey_local_threshold_calibration.csv`.